## Structured Data Cleaning

**Architecture Overview:**
This notebook executes an industrial-grade data cleaning and synchronization task. We read Parquet/Delta files directly from the MinIO (S3) data lake using Apache Spark, perform deep structural cleaning in distributed memory (primary key hardening, anomaly handling, dynamic field inference), and finally synchronize the data to the ClickHouse modern analytical data warehouse via high-concurrency direct writes (MapPartitions).

**Pipeline Steps:**
1. **Environment Initialization**: Mount the Spark Session and connect to MinIO and ClickHouse.
2. **Dynamic Metadata Scanning**: Automatically identify valid data directories in the S3 Landing Zone.
3. **Distributed Adaptive Cleaning**: Apply different deduplication rules for different business tables (e.g., full-field deduplication for vehicle tables, composite primary key deduplication for climate tables).
4. **High-Concurrency Ingestion**: Pre-build DDL on the Driver, and perform distributed direct writes to ClickHouse from the Executors.
5. **Post-Cleaning Validation**: Check physical metrics, data distribution, and real sample exploration.

**Production synchronization note: structured cleaning rules and quarantine.**

The production Trusted Zone DAG now declares dataset-specific quality rules for `co2_emission_by_vehicles`, `global_warming_dataset`, `natural_disaster_tweets`, and `temperature_change`: expected columns, required columns, type/range normalization, deduplication behavior, and per-dataset schema versions. Rows or datasets that fail required checks are excluded from trusted ClickHouse business tables and persisted as rejected/quarantine evidence under `s3://trusted-zone/rejected/structured/`. This notebook remains the exploratory/runbook version of the same logic, while the DAG owns the reproducible rejected-path implementation.


**Importing Useful Libraries**

In [1]:
import os
import re
import ast
import boto3
import clickhouse_connect
from datetime import datetime, timezone
import pyspark.sql.functions as F
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import StringType, IntegerType, LongType, FloatType, DoubleType, DateType, TimestampType, ArrayType
from dotenv import load_dotenv

# 1. Load environment variables
load_dotenv()
endpoint = os.getenv("MINIO_ENDPOINT")
MINIO_ROLE = "reader"
if MINIO_ROLE == "admin":
    access_key = os.getenv("MINIO_ACCESS_KEY")
    secret_key = os.getenv("MINIO_SECRET_KEY")
else:
    role_prefix = MINIO_ROLE.upper()
    access_key = os.getenv(f"MINIO_{role_prefix}_ACCESS_KEY")
    secret_key = os.getenv(f"MINIO_{role_prefix}_SECRET_KEY")
if not endpoint or not access_key or not secret_key:
    raise RuntimeError(f"Missing MinIO {MINIO_ROLE} credentials in environment")
TRUSTED_LOGICAL_DATE = os.getenv("TRUSTED_LOGICAL_DATE") or datetime.now(timezone.utc).isoformat()

# 2. Initialize Spark Session with S3 & Delta bindings
DELTA_VERSION = "4.1.0" 

spark = SparkSession.builder \
    .appName("Production-Data-Warehouse-Pipeline") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages", 
            f"org.apache.hadoop:hadoop-aws:3.3.4,"
            f"com.amazonaws:aws-java-sdk-bundle:1.12.262,"
            f"io.delta:delta-spark_2.13:{DELTA_VERSION}") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", endpoint) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.access.key", access_key) \
    .config("spark.hadoop.fs.s3a.secret.key", secret_key) \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

# 3. Normalize Hadoop config values to prevent JVM parse errors
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
for item in hadoop_conf.iterator():
    key, value = item.getKey(), item.getValue()
    if isinstance(value, str) and (value.endswith("s") or value.endswith("h")):
        hadoop_conf.set(key, "".join(char for char in value if char.isdigit()))

print("Spark Cluster environment initialized successfully.")

Spark Cluster environment initialized successfully.


In [2]:
STRUCTURED_BASE_PATH = "s3a://landing-zone/persistent-landing/structured/"
EXCLUDED_DATASET_FOLDERS = {"raw", "file_catalog"}


def discover_structured_dataset_paths(spark_session: SparkSession, base_path: str = STRUCTURED_BASE_PATH) -> list[str]:
    """Return valid structured dataset directories from the landing-zone path."""
    sc = spark_session.sparkContext
    path_obj = sc._jvm.org.apache.hadoop.fs.Path(base_path)
    fs = path_obj.getFileSystem(sc._jsc.hadoopConfiguration())

    dataset_paths = []
    for status in fs.listStatus(path_obj):
        if not status.isDirectory():
            continue

        full_path = status.getPath().toString()
        folder_name = full_path.rstrip("/").split("/")[-1]
        if folder_name in EXCLUDED_DATASET_FOLDERS or folder_name.startswith("."):
            continue

        dataset_paths.append(full_path)

    return sorted(dataset_paths)


valid_delta_paths = discover_structured_dataset_paths(spark)
print(f"Discovered {len(valid_delta_paths)} analytical dataset targets in S3 Landing Zone.")


Discovered 4 analytical dataset targets in S3 Landing Zone.


### Core Cleaning Functions
The following cells split the trusted-zone processing into focused helpers: source reading, table and column normalization, baseline data cleaning, ClickHouse DDL generation, distributed loading, and validation.

In [3]:
from functools import reduce
from pyspark.sql.types import BooleanType, ShortType, ByteType

NULL_TOKENS = {"", "na", "n/a", "null", "none", "nan", "-", "--"}
DEGREE_C = chr(0x00b0) + "C"
CORRUPTED_DEGREE_C_PATTERN = f"(?i)[{chr(0x00e2)}{chr(0x00c2)}]\\s*[{chr(0x00b0)}{chr(0x00ba)}]\\s*c"
CLEAN_DEGREE_C_PATTERN = f"(?i){chr(0x00b0)}\\s*c"
MOJIBAKE_DASH_PATTERN = f"{chr(0x00e2)}{chr(0x0080)}[{chr(0x0093)}{chr(0x0094)}]"
ARRAY_LIKE_COLUMNS = {"hashtags", "emojis"}

TABLE_SORTING_KEYS = {
    "global_warming": ["country", "year"],
    "temperature_change": ["area", "months", "year"],
    "emission": ["make", "model", "vehicle_class"],
    "tweet": ["id"],
}

TRUSTED_SCHEMA_VERSION = "trusted_v1"
STRUCTURED_DATASET_RULES = {
    "co2_emission_by_vehicles": {"schema_version": "co2_emission_by_vehicles_v1", "required_columns": ["make", "model", "vehicle_class"]},
    "global_warming_dataset": {"schema_version": "global_warming_dataset_v1", "required_columns": ["country", "year"]},
    "natural_disaster_tweets": {"schema_version": "natural_disaster_tweets_v1", "required_columns": ["id"]},
    "temperature_change": {"schema_version": "temperature_change_v1", "required_columns": ["area", "months", "year"]},
}


def rules_for_table(table_name: str) -> dict:
    for table_hint, rules in STRUCTURED_DATASET_RULES.items():
        if table_hint in table_name:
            return rules
    return {"schema_version": TRUSTED_SCHEMA_VERSION, "required_columns": []}


def required_columns_for_table(table_name: str) -> list[str]:
    return list(rules_for_table(table_name).get("required_columns", []))


def schema_version_for_table(table_name: str) -> str:
    return rules_for_table(table_name).get("schema_version", TRUSTED_SCHEMA_VERSION)


def validate_required_columns(df: DataFrame, table_name: str) -> None:
    missing = [column for column in required_columns_for_table(table_name) if column not in df.columns]
    if missing:
        raise ValueError(f"{table_name} is missing required columns: {missing}")


def add_governance_metadata(df: DataFrame, source_path: str) -> DataFrame:
    return (
        df.withColumn("source_system", F.lit("landing-zone"))
        .withColumn("ingestion_time", F.lit(TRUSTED_LOGICAL_DATE))
        .withColumn("source_file_path", F.lit(source_path))
        .withColumn("validation_status", F.lit("valid"))
        .withColumn("schema_version", F.lit(schema_version_for_table(normalize_table_name(source_path))))
    )

CLICKHOUSE_TYPE_MAP = {
    StringType: "String",
    IntegerType: "Int32",
    LongType: "Int64",
    ShortType: "Int16",
    ByteType: "Int8",
    FloatType: "Float32",
    DoubleType: "Float64",
    BooleanType: "UInt8",
    DateType: "Date",
    TimestampType: "DateTime",
}


def normalize_table_name(s3_path: str) -> str:
    """Convert a lake directory name into a stable warehouse table name."""
    folder_name = s3_path.rstrip("/").split("/")[-1]
    table_name = re.sub(r"_\d+$", "", folder_name.replace("_delta", ""))
    return re.sub(r"_+", "_", table_name.replace("-", "_").lower()).strip("_")


def read_delta_or_parquet(spark_session: SparkSession, s3_path: str) -> DataFrame:
    """Read Delta when a _delta_log exists; otherwise fall back to Parquet."""
    sc = spark_session.sparkContext
    delta_log_path = sc._jvm.org.apache.hadoop.fs.Path(s3_path.rstrip("/") + "/_delta_log")
    fs = delta_log_path.getFileSystem(sc._jsc.hadoopConfiguration())
    reader_format = "delta" if fs.exists(delta_log_path) else "parquet"
    return spark_session.read.format(reader_format).load(s3_path)


def normalize_column_name(column_name: str, position: int) -> str:
    """Build a ClickHouse-friendly column name while preserving readable business terms."""
    cleaned = column_name.replace("\ufeff", "")
    cleaned = re.sub(r"[()]+", "", cleaned)
    cleaned = re.sub(r"[^0-9A-Za-z_]+", "_", cleaned.strip())
    cleaned = re.sub(r"_+", "_", cleaned).strip("_")
    cleaned = cleaned or f"column_{position + 1}"

    if cleaned.lower() in {"tweet_id", "id"}:
        cleaned = "id"
    if cleaned[0].isdigit():
        cleaned = f"col_{cleaned}"

    # ClickHouse identifiers are standardized to lowercase for downstream SQL stability.
    return cleaned.lower()


def standardize_column_names(df: DataFrame) -> DataFrame:
    """Normalize headers and make duplicate names deterministic."""
    seen = {}
    renamed_df = df

    for idx, original_name in enumerate(df.columns):
        base_name = normalize_column_name(original_name, idx)
        occurrence = seen.get(base_name.lower(), 0)
        seen[base_name.lower()] = occurrence + 1
        final_name = base_name if occurrence == 0 else f"{base_name}_{occurrence + 1}"

        if final_name != original_name:
            renamed_df = renamed_df.withColumnRenamed(original_name, final_name)

    return renamed_df


In [4]:
def normalize_string_columns(df: DataFrame, table_name: str) -> DataFrame:
    """Clean string columns for trusted-zone use: controls, whitespace, null tokens, and Celsius mojibake."""
    string_columns = [field.name for field in df.schema.fields if isinstance(field.dataType, StringType)]

    for column_name in string_columns:
        cleaned_col = F.regexp_replace(F.col(column_name), r"[\n\r\t]", " ")
        cleaned_col = F.regexp_replace(cleaned_col, r"\s+", " ")
        cleaned_col = F.regexp_replace(cleaned_col, MOJIBAKE_DASH_PATTERN, "-")
        cleaned_col = F.regexp_replace(cleaned_col, CORRUPTED_DEGREE_C_PATTERN, DEGREE_C)
        cleaned_col = F.regexp_replace(cleaned_col, CLEAN_DEGREE_C_PATTERN, DEGREE_C)
        cleaned_col = F.regexp_replace(cleaned_col, r"[\u0000-\u0008\u000B\u000C\u000E-\u001F\u007F-\u009F]", "")
        trimmed_col = F.trim(cleaned_col)
        lower_trimmed_col = F.lower(trimmed_col)

        if table_name == "temperature_change" and column_name == "unit":
            normalized_col = F.when(
                lower_trimmed_col.isin("c", "celsius", "degree c", "degrees c"),
                F.lit(DEGREE_C),
            ).otherwise(trimmed_col)
        else:
            normalized_col = lower_trimmed_col

        df = df.withColumn(
            column_name,
            F.when(lower_trimmed_col.isin(*sorted(NULL_TOKENS)), F.lit(None)).otherwise(normalized_col),
        )

    return df


def drop_empty_rows(df: DataFrame) -> DataFrame:
    """Remove rows where every field is null or blank."""
    if not df.columns:
        return df

    non_empty_checks = []
    for field in df.schema.fields:
        col_ref = F.col(field.name)
        if isinstance(field.dataType, StringType):
            non_empty_checks.append(col_ref.isNotNull() & (F.trim(col_ref) != ""))
        else:
            non_empty_checks.append(col_ref.isNotNull())

    return df.where(reduce(lambda left, right: left | right, non_empty_checks))


def mask_invalid_numeric_values(df: DataFrame) -> DataFrame:
    """Convert NaN and Infinity values to null before warehouse ingestion."""
    for field in df.schema.fields:
        if isinstance(field.dataType, (FloatType, DoubleType)):
            as_text = F.lower(F.col(field.name).cast("string"))
            df = df.withColumn(
                field.name,
                F.when(F.isnan(F.col(field.name)) | as_text.isin("infinity", "+infinity", "-infinity", "inf", "+inf", "-inf"), F.lit(None))
                 .otherwise(F.col(field.name)),
            )
    return df


def normalize_array_columns(df: DataFrame) -> DataFrame:
    """Normalize known array-like columns that may arrive as Python-list strings."""
    dtype_lookup = dict(df.dtypes)

    for column_name in ARRAY_LIKE_COLUMNS.intersection(df.columns):
        if "string" in dtype_lookup[column_name]:
            df = df.withColumn(
                column_name,
                F.split(F.regexp_replace(F.col(column_name), r'[\[\]\'"\s]', ""), ","),
            )

        df = df.withColumn(
            column_name,
            F.expr(f"filter(transform({column_name}, x -> lower(trim(x))), x -> x IS NOT NULL AND x != '')"),
        )

    return df


def apply_table_specific_fixes(df: DataFrame, table_name: str) -> DataFrame:
    """Keep table-specific fixes isolated from general trusted-zone cleaning."""
    if "id" in df.columns and "tweet" in table_name:
        df = df.withColumn("id", F.col("id").cast("string"))

    return df


def apply_trusted_zone_cleaning(df: DataFrame, table_name: str) -> DataFrame:
    """Apply baseline trusted-zone cleaning before table-specific warehouse loading."""
    df = standardize_column_names(df)
    df = apply_table_specific_fixes(df, table_name)
    df = drop_empty_rows(df)
    df = normalize_string_columns(df, table_name)
    df = mask_invalid_numeric_values(df)
    df = normalize_array_columns(df)
    return df


In [5]:
def choose_sorting_keys(df: DataFrame, table_name: str) -> list[str]:
    """Choose stable ClickHouse MergeTree sorting keys for each known business table."""
    sorting_keys = []
    for table_hint, candidate_keys in TABLE_SORTING_KEYS.items():
        if table_hint in table_name:
            sorting_keys = candidate_keys
            break

    final_keys = [key for key in sorting_keys if key in df.columns]
    if not final_keys and "id" in df.columns:
        final_keys = ["id"]
    if not final_keys and df.columns:
        final_keys = [df.columns[0]]

    return final_keys


def fill_sorting_key_nulls(df: DataFrame, sorting_keys: list[str]) -> DataFrame:
    """ClickHouse sorting keys cannot be nullable, so fill only those columns."""
    dtype_lookup = dict(df.dtypes)

    for column_name in sorting_keys:
        column_type = dtype_lookup[column_name]
        if "string" in column_type:
            df = df.withColumn(column_name, F.coalesce(F.col(column_name), F.lit("unknown")))
        elif any(token in column_type for token in ["int", "long", "short", "byte"]):
            df = df.withColumn(column_name, F.coalesce(F.col(column_name), F.lit(0)))
        elif "double" in column_type or "float" in column_type:
            df = df.withColumn(column_name, F.coalesce(F.col(column_name), F.lit(0.0)))

    return df


def deduplicate_for_trusted_zone(df: DataFrame, sorting_keys: list[str], table_name: str) -> DataFrame:
    """Use conservative deduplication: known source tables by full row, otherwise by sorting keys."""
    known_business_table = any(token in table_name for token in ["emission", "global_warming", "temperature_change", "tweet"])
    if known_business_table:
        return df.dropDuplicates()
    if sorting_keys:
        return df.dropDuplicates(subset=sorting_keys)
    return df.dropDuplicates()


def clickhouse_type_for_field(field) -> str:
    """Translate Spark field types into ClickHouse column types."""
    if isinstance(field.dataType, ArrayType):
        element_type = CLICKHOUSE_TYPE_MAP.get(type(field.dataType.elementType), "String")
        return f"Array({element_type})"
    return CLICKHOUSE_TYPE_MAP.get(type(field.dataType), "String")


def generate_clickhouse_ddl(df: DataFrame, table_name: str, sorting_keys: list[str], database_name: str) -> str:
    """Generate a MergeTree DDL from the cleaned Spark schema."""
    ch_columns = []
    sorting_key_set = set(sorting_keys)

    for field in df.limit(0).schema.fields:
        ch_type = clickhouse_type_for_field(field)
        is_array = isinstance(field.dataType, ArrayType)
        if field.name in sorting_key_set or is_array:
            ch_columns.append(f"    `{field.name}` {ch_type}")
        else:
            ch_columns.append(f"    `{field.name}` Nullable({ch_type})")

    order_by_clause = ", ".join([f"`{key}`" for key in sorting_keys]) if sorting_keys else f"`{df.columns[0]}`"
    return (
        f"CREATE TABLE IF NOT EXISTS {database_name}.{table_name} (\n"
        f"{',\n'.join(ch_columns)}\n"
        ") ENGINE = MergeTree()\n"
        f"ORDER BY ({order_by_clause})"
    )


def extract_clean_dataframe_and_ddl(spark_session: SparkSession, s3_path: str, database_name: str = "bi_analytics") -> tuple[DataFrame, str]:
    """Read a structured source, clean it for the trusted zone, and build ClickHouse DDL."""
    table_name = normalize_table_name(s3_path)
    df = read_delta_or_parquet(spark_session, s3_path)
    cleaned_df = apply_trusted_zone_cleaning(df, table_name)
    validate_required_columns(cleaned_df, table_name)
    sorting_keys = choose_sorting_keys(cleaned_df, table_name)
    cleaned_df = fill_sorting_key_nulls(cleaned_df, sorting_keys)
    cleaned_df = deduplicate_for_trusted_zone(cleaned_df, sorting_keys, table_name)
    cleaned_df = add_governance_metadata(cleaned_df, s3_path)
    ddl_sql = generate_clickhouse_ddl(cleaned_df, table_name, sorting_keys, database_name)
    return cleaned_df, ddl_sql


In [6]:
def load_dataframe_to_clickhouse_parallel(df: DataFrame, ddl_sql: str, target_table_name: str, database_name: str = "bi_analytics"):
    """Use Spark workers to write cleaned partitions directly into ClickHouse."""
    driver_client = clickhouse_connect.get_client(
        host="clickhouse",
        port=8123,
        username=os.getenv("CLICKHOUSE_TRUSTED_USER", os.getenv("CLICKHOUSE_USER", "analytics")),
        password=os.getenv("CLICKHOUSE_TRUSTED_PASSWORD", os.getenv("CLICKHOUSE_PASSWORD", "analytics_secret")),
        database=database_name,
    )
    driver_client.command(f"DROP TABLE IF EXISTS {database_name}.{target_table_name}")
    driver_client.command(ddl_sql)
    driver_client.close()

    column_names = df.columns

    def write_partition(rows_iter):
        worker_client = clickhouse_connect.get_client(
            host="clickhouse",
            port=8123,
            username=os.getenv("CLICKHOUSE_TRUSTED_USER", os.getenv("CLICKHOUSE_USER", "analytics")),
            password=os.getenv("CLICKHOUSE_TRUSTED_PASSWORD", os.getenv("CLICKHOUSE_PASSWORD", "analytics_secret")),
            database=database_name,
        )
        payload = [tuple(row) for row in rows_iter]
        if payload:
            worker_client.insert(table=target_table_name, data=payload, column_names=column_names)
        worker_client.close()
        return [len(payload)]

    total_inserted = df.rdd.mapPartitions(write_partition).sum()
    print(f"Parallel sync completed for '{target_table_name}'. Total rows ingested: {total_inserted}")


In [7]:
def process_structured_dataset(s3_path: str, database_name: str = "bi_analytics") -> dict:
    """Clean one structured dataset and sync it into ClickHouse."""
    table_name = normalize_table_name(s3_path)
    print(f"\nProcessing target: {table_name} ...")

    cleaned_df, clickhouse_ddl = extract_clean_dataframe_and_ddl(spark, s3_path, database_name=database_name)
    load_dataframe_to_clickhouse_parallel(cleaned_df, clickhouse_ddl, target_table_name=table_name, database_name=database_name)
    return {"table": table_name, "status": "success"}


def run_structured_cleaning_pipeline(dataset_paths: list[str]) -> list[dict]:
    """Run the full trusted-zone cleaning and warehouse ingestion pipeline."""
    print("Starting Automated Data Cleaning & Warehouse Ingestion Pipeline")
    results = []

    for s3_path in dataset_paths:
        table_name = normalize_table_name(s3_path)
        try:
            results.append(process_structured_dataset(s3_path))
        except Exception as exc:
            print(f"Synchronizer halted processing target table '{table_name}'. Error: {exc}")
            results.append({"table": table_name, "status": "failed", "error": str(exc)})

    successful = sum(1 for result in results if result["status"] == "success")
    print(f"\nStructured cleaning pipeline completed: {successful}/{len(results)} tables synced successfully.")
    return results


pipeline_results = run_structured_cleaning_pipeline(valid_delta_paths)
pipeline_results


Starting Automated Data Cleaning & Warehouse Ingestion Pipeline

Processing target: co2_emission_by_vehicles ...
Parallel sync completed for 'co2_emission_by_vehicles'. Total rows ingested: 5988

Processing target: global_warming_dataset ...
Parallel sync completed for 'global_warming_dataset'. Total rows ingested: 100000

Processing target: natural_disaster_tweets ...
Parallel sync completed for 'natural_disaster_tweets'. Total rows ingested: 127527

Processing target: temperature_change ...
Parallel sync completed for 'temperature_change'. Total rows ingested: 241893

Structured cleaning pipeline completed: 4/4 tables synced successfully.


[{'table': 'co2_emission_by_vehicles', 'status': 'success'},
 {'table': 'global_warming_dataset', 'status': 'success'},
 {'table': 'natural_disaster_tweets', 'status': 'success'},
 {'table': 'temperature_change', 'status': 'success'}]

### Post-Cleaning Data Validation
This testing module connects to the final ClickHouse physical data warehouse layer to verify:
1. **Physical Storage Metrics**: Check whether the stored row count has severely shrunk/expanded, and monitor disk space utilization.
2. **Data Sampling**: Visually verify that composite primary keys and nested arrays meet ClickHouse's business requirements.

In [8]:
import pandas as pd


def get_clickhouse_client(database_name: str = "bi_analytics"):
    """Create a ClickHouse client for validation queries."""
    return clickhouse_connect.get_client(
        host="clickhouse",
        port=8123,
        username="analytics",
        password="analytics_secret",
        database=database_name,
    )


def print_storage_metrics(client, database_name: str = "bi_analytics") -> None:
    """Print physical row counts and storage size for active ClickHouse parts."""
    metrics_query = f"""
    SELECT
        table AS table_name,
        sum(rows) AS total_physical_rows,
        formatReadableSize(sum(bytes_on_disk)) AS disk_usage_size
    FROM system.parts
    WHERE database = '{database_name}' AND active = 1
    GROUP BY table
    ORDER BY table
    """
    print("Data Warehouse Storage Metrics (ClickHouse)")
    for row in client.query(metrics_query).result_rows:
        print(f"Table: {row[0]:<30} | Rows: {row[1]:<10} | Disk Storage: {row[2]}")


def check_temperature_unit_encoding(client, database_name: str = "bi_analytics") -> None:
    """Verify that temperature_change.unit no longer contains mojibake markers."""
    bad_marker_1 = chr(0x00e2)
    bad_marker_2 = chr(0x00c2)
    unit_check_query = f"""
    SELECT unit, count()
    FROM {database_name}.temperature_change
    WHERE position(unit, '{bad_marker_1}') > 0 OR position(unit, '{bad_marker_2}') > 0
    GROUP BY unit
    """
    bad_units = client.query(unit_check_query).result_rows
    if bad_units:
        print("WARNING: Remaining mojibake unit values detected:", bad_units)
    else:
        print("Unit encoding check passed: no mojibake values found in temperature_change.unit.")


def preview_target_tables(client, target_tables: list[str], database_name: str = "bi_analytics", sample_size: int = 3) -> None:
    """Display small untruncated samples from each target table."""
    pd.set_option("display.max_columns", None)
    pd.set_option("display.max_colwidth", None)
    pd.set_option("display.width", 1000)

    print("Target Table Data Sampling Preview (Top 3 Rows - Untruncated)")
    for table in target_tables:
        try:
            headers_query = f"SELECT name FROM system.columns WHERE database = '{database_name}' AND table = '{table}' ORDER BY position"
            headers = [row[0] for row in client.query(headers_query).result_rows]
            data_res = client.query(f"SELECT * FROM {database_name}.{table} LIMIT {sample_size}")

            print(f"\nTable: {table}")
            print("-" * 80)
            if not data_res.result_rows:
                print("  Empty: No data available in ClickHouse.")
                continue

            pdf = pd.DataFrame(data_res.result_rows, columns=headers)
            for column_name in pdf.columns:
                pdf[column_name] = pdf[column_name].apply(lambda value: list(value) if isinstance(value, (list, tuple)) else value)
            display(pdf)

            if table == "temperature_change" and "unit" in headers:
                check_temperature_unit_encoding(client, database_name=database_name)

        except Exception as exc:
            print(f"Could not preview table '{table}': {exc}")


client = get_clickhouse_client()
try:
    print_storage_metrics(client)
    preview_target_tables(
        client,
        ["natural_disaster_tweets", "co2_emission_by_vehicles", "temperature_change", "global_warming_dataset"],
    )
finally:
    client.close()


Data Warehouse Storage Metrics (ClickHouse)
Table: co2_emission_by_vehicles       | Rows: 5988       | Disk Storage: 133.06 KiB
Table: global_warming_dataset         | Rows: 100000     | Disk Storage: 18.12 MiB
Table: natural_disaster_tweets        | Rows: 127527     | Disk Storage: 12.83 MiB
Table: temperature_change             | Rows: 241893     | Disk Storage: 2.22 MiB
Target Table Data Sampling Preview (Top 3 Rows - Untruncated)

Table: natural_disaster_tweets
--------------------------------------------------------------------------------


,id,tweet_text,disaster_type,hashtags,emojis
0,1.00113669658963149e18,"flash floods struck a maryland city on sunday, washing out streets and tossing cars like bath toys.",flood,[],[]
1,1.0011369503451095e18,state of emergency declared for maryland flooding: via @youtube,flood,[],[]
2,1.00113733405683302e18,"other parts of maryland also saw significant damage from sundays storms including this baltimore city neighborhood, #dundalk and #catonsville. rain totals spanned from 1 to 10 inches across maryland: #ecflood",flood,"[dundalk, catonsville, ecflood]",[]



Table: co2_emission_by_vehicles
--------------------------------------------------------------------------------


,make,model,vehicle_class,engine_sizel,cylinders,transmission,fuel_type,fuel_consumption_city_l_100_km,fuel_consumption_hwy_l_100_km,fuel_consumption_comb_l_100_km,fuel_consumption_comb_mpg,co2_emissionsg_km
0,acura,ilx,compact,2.4,4,am8,z,9.3,6.6,8.1,35,189
1,acura,ilx,compact,2.0,4,as5,z,9.9,6.7,8.5,33,196
2,acura,ilx,compact,2.4,4,am8,z,9.4,6.8,8.2,34,192



Table: temperature_change
--------------------------------------------------------------------------------


,domain_code,domain,area_code_m49,area,element_code,element,months_code,months,year_code,year,unit,value,flag,flag_description
0,et,temperature change on land,4,afghanistan,7271,temperature change,7004,april,1962,1962,°C,0.040,e,estimated value
1,et,temperature change on land,4,afghanistan,7271,temperature change,7004,april,1963,1963,°C,0.859,e,estimated value
2,et,temperature change on land,4,afghanistan,7271,temperature change,7004,april,1966,1966,°C,-1.101,e,estimated value


Unit encoding check passed: no mojibake values found in temperature_change.unit.

Table: global_warming_dataset
--------------------------------------------------------------------------------


,country,year,temperature_anomaly,co2_emissions,population,forest_area,gdp,renewable_energy_usage,methane_emissions,sea_level_rise,arctic_ice_extent,urbanization,deforestation_rate,extreme_weather_events,average_rainfall,solar_energy_potential,waste_management,per_capita_emissions,industrial_activity,air_pollution_index,biodiversity_index,ocean_acidification,fossil_fuel_usage,energy_consumption_per_capita,policy_score,average_temperature
0,country_1,1900,0.772039,2.111143e+08,5.576762e+08,14.401780,3.466274e+12,46.673056,6.547845e+06,42.738115,11.447361,20.752819,2.034420,10,2355.376329,986.819604,91.111111,18.943434,69.598433,150.584727,79.475557,8.067704,57.699551,3518.096248,60.284283,13.235857
1,country_1,1902,-1.110650,5.300647e+08,2.669385e+08,46.563315,2.931235e+12,97.924409,9.075092e+06,27.097804,3.577470,72.857078,0.149165,14,3258.275871,1702.377746,66.496025,4.076063,56.633721,184.588546,60.884943,7.802114,61.645229,738.224252,76.041174,21.678715
2,country_1,1904,-1.319920,9.619559e+08,7.383737e+08,39.551921,4.918382e+12,57.245417,3.647545e+06,0.858311,8.809201,62.026295,1.911535,17,2036.448479,1694.309523,25.127089,2.086132,57.563322,185.585293,51.555234,7.702761,48.880101,4328.604536,63.683835,24.823660


In [9]:
spark.stop()